# Navigator HITL Verification (Real Data)

This notebook verifies the **Query Navigator** as a **Clue Generator**.

### Verification Workflow:
1. **Full Ingestion:** Run the `IngestionPipeline` to generate `schema_summary.json` and `ufl.parquet`.
2. **Navigate:** Verify the Navigator generates precise clues (Metrics, Entities, Years) without overstepping into routing decisions.

In [1]:
%load_ext autoreload
%autoreload 2

import sys
import os
import nest_asyncio
from dotenv import load_dotenv

# Add src to path
sys.path.append(os.path.abspath("../src"))
load_dotenv("../.env")

from venra.pipeline import IngestionPipeline
from venra.navigator import Navigator
from venra.logging_config import logger

nest_asyncio.apply()

/Users/pedram/Projects/VeNRA/.venv/lib/python3.11/site-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'validate_default' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'validate_default' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(


## 1. Prepare Data (Ingestion)

This ensures `ufl.parquet` and `schema_summary.json` are created correctly.

In [2]:
# WARNING: TAKES TIME (>30 MIn)
PDF_PATH = "../data/10K_TD_test.pdf"
pipeline = IngestionPipeline()
await pipeline.run(PDF_PATH, skip_parsing=True)

2026-02-23 22:32:04,817 - venra - INFO - Loading existing DOM from /Users/pedram/Projects/VeNRA/data/processed/10K_TD_test_dom.pkl
2026-02-23 22:32:04,820 - venra - INFO - Resolving Entity from Cover Page context…
2026-02-23 22:32:05,762 - venra - INFO - Resolved Entity: ID_TDG (TransDigm Group Incorporated)
2026-02-23 22:32:05,763 - venra - INFO - Using Global Context: Registrant: TransDigm Group Incorporated. Current Fiscal Year: 2025. Dollars in millions unless specified.
2026-02-23 22:32:08,389 - venra - INFO - Indexed 61 blocks in ChromaDB.
2026-02-23 22:32:08,390 - venra - INFO - Extracting facts from text in ['☒ Annual Report Pursuant to Section 13 or 15(d) of the Securities Exchange Act of 1934']...
2026-02-23 22:32:09,426 - venra - INFO - Extracting facts from text in ['☐ Transition Report pursuant to Section 13 or 15(d) of the Securities Exchange Act of 1934']...
2026-02-23 22:32:10,253 - venra - WARNING - Semantic Rejection: Metric 'Revenue' tokens ['revenue'] not grounded i

[UFLRow(row_id='2580551c8f3db22b13c690fda656baf1', canonical_entity_id='ID_TDG', entity_name_raw='TransDigm Group Incorporated', metric_name='Revenue', related_entity_id=None, num_value=1000000000.0, grounding_quote='$1 billion', unit_normalized='USD', scale=1.0, period_start=None, period_end='2025', period_type=None, doc_section='☒ Annual Report Pursuant to Section 13 or 15(d) of the Securities Exchange Act of 1934', source_chunk_id='721dcecfe0c72bc3e41ca4903c2491f5', text_nuance=None, char_interval=None, alignment_status='UNALIGNED', confidence_score=0.0),
 UFLRow(row_id='c0f593bba15a8c08de28a08e885aaa64', canonical_entity_id='ID_TDG', entity_name_raw='TransDigm Group Incorporated', metric_name='Revenue', related_entity_id=None, num_value=None, grounding_quote='For the fiscal year ended September 30, 2025', unit_normalized='N/A', scale=1.0, period_start=None, period_end='2025-09-30', period_type=None, doc_section='☐ Transition Report pursuant to Section 13 or 15(d) of the Securities 

## 2. Initialize Navigator

The Navigator loads the prompt and the schema context generated above.

In [3]:
file_prefix = os.path.basename(PDF_PATH).replace(".pdf", "")
nav = Navigator(file_prefix=file_prefix)

## 3. Live Navigation (Actual LLM Calls)

Verify that the Navigator maps intent to clues correctly.

In [4]:
query = "How much did the company spend on acquisitions in 2025?"
plan = await nav.navigate(query)

print(f"--- Clues for: {query} ---")
print(f"Reasoning: {plan.reasoning}")
if plan.ufl_query:
    print(f"Entities:  {plan.ufl_query.entity_ids}")
    print(f"Metrics:   {plan.ufl_query.metric_keywords}")
    print(f"Years:     {plan.ufl_query.years}")
print(f"Hypothesis: {plan.vector_hypothesis}")

2026-02-23 23:19:08,018 - venra - INFO - Navigating query: How much did the company spend on acquisitions in 2025?
2026-02-23 23:19:08,568 - venra - INFO - Plan generated. Reasoning: The user is asking for the amount spent on acquisitions by TransDigm Group Incorporated in 2025. We will search for the entity ID 'ID_TDG' and the metric 'Acquisitions' in the 2025 data.
--- Clues for: How much did the company spend on acquisitions in 2025? ---
Reasoning: The user is asking for the amount spent on acquisitions by TransDigm Group Incorporated in 2025. We will search for the entity ID 'ID_TDG' and the metric 'Acquisitions' in the 2025 data.
Entities:  ['ID_TDG']
Metrics:   ['Acquisitions']
Years:     ['2025']
Hypothesis: Acquisitions in 2025
